In [53]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/evelynuntariady/thesis/data_thesis_final.csv
/kaggle/input/datasets/evelynuntariady/thesis/raw_data_all.csv
/kaggle/input/datasets/evelynuntariady/thesis/thesis_full.csv
/kaggle/input/datasets/evelynuntariady/thesis/data_thesis_majority_vote.csv


In [54]:
! pip install -q optuna
! pip install -q emoji

In [55]:
import pandas as pd
import numpy as np
import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from huggingface_hub import login
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
import optuna
import ast
import re
import emoji
from transformers.utils import logging as hf_logging
import gc
import shutil
import random
import math

In [56]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [57]:
# df_test = pd.read_csv("/kaggle/input/datasets/evelynuntariady/thesis/data_thesis_final.csv")

In [58]:
df_main = pd.read_csv('/kaggle/input/datasets/evelynuntariady/thesis/thesis_full.csv')

In [59]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27833 entries, 0 to 27832
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   text_id                        27833 non-null  object
 1   annotators_id                  27833 non-null  object
 2   text                           27832 non-null  object
 3   topic                          27833 non-null  object
 4   is_noise_or_spam_text          27833 non-null  object
 5   toxicity                       27833 non-null  object
 6   polarized                      27833 non-null  object
 7   profanity_obscenity            27833 non-null  object
 8   threat_incitement_to_violence  27833 non-null  object
 9   insults                        27833 non-null  object
 10  identity_attack                27833 non-null  object
 11  sexually_explicit              27833 non-null  object
 12  key                            27833 non-null  object
 13  c

In [62]:
cols = [
'is_noise_or_spam_text',
'toxicity',
'polarized',
'profanity_obscenity',
'threat_incitement_to_violence',
'insults',
'identity_attack',
'sexually_explicit'
]

for col in cols:
    df_main[col] = df_main[col].apply(lambda x: ast.literal_eval(x) if isinstance(x,str) else x)

In [63]:
print(df_main['is_noise_or_spam_text'].iloc[0])
print(type(df_main['is_noise_or_spam_text'].iloc[0]))

print(df_main['polarized'].iloc[0])
print(type(df_main['polarized'].iloc[0]))

print(df_main['profanity_obscenity'].iloc[0])
print(type(df_main['profanity_obscenity'].iloc[0]))

print(df_main['insults'].iloc[0])
print(type(df_main['insults'].iloc[0]))

print(df_main['sexually_explicit'].iloc[0])
print(type(df_main['sexually_explicit'].iloc[0]))

print(df_main['identity_attack'].iloc[0])
print(type(df_main['identity_attack'].iloc[0]))

print(df_main['toxicity'].iloc[0])
print(type(df_main['toxicity'].iloc[0]))

['0', '0']
<class 'list'>
['0', '0']
<class 'list'>
['0', '0']
<class 'list'>
['0', '0']
<class 'list'>
['0', '0']
<class 'list'>
['0', '0']
<class 'list'>
['0', '0']
<class 'list'>


In [64]:
df_main.iloc[5389]

text_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           1-6084
annotators_id                                                                                                                                                                                                                                                  

In [65]:
df_main['is_noise_or_spam_text'].iloc[5389] = [0, 1, 0]

/tmp/ipykernel_48/1462503684.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_main['is_noise_or_spam_text'].iloc[5389] = [0, 1, 0]


In [66]:
for col in cols:
    for idx, x in df_main[col].items():
        try:
            for i in x:
                int(i)
        except Exception as e:
            print(f"Column: {col}")
            print(f"Row: {idx}")
            print("Cell:", x)
            print("Cell type:", type(x))
            print("Element types:", [type(i) for i in x])
            raise

### MAJORITY VOTE WO POLARIZED

In [67]:
print(cols)
cols2 = cols
cols2.remove('polarized')
print(cols2)

['is_noise_or_spam_text', 'toxicity', 'polarized', 'profanity_obscenity', 'threat_incitement_to_violence', 'insults', 'identity_attack', 'sexually_explicit']
['is_noise_or_spam_text', 'toxicity', 'profanity_obscenity', 'threat_incitement_to_violence', 'insults', 'identity_attack', 'sexually_explicit']


In [68]:
def majority_vote(x):
    x = [int(i) for i in x]
    return round(sum(x)/len(x))

for col in cols2:
    df_main[col] = df_main[col].apply(majority_vote)

In [ ]:
df_main = df_main[df_main['is_noise_or_spam_text'] == 0]
df_main = df_main.dropna()
df_main.isnull().sum()

text_id                          0
annotators_id                    0
text                             0
topic                            0
is_noise_or_spam_text            0
toxicity                         0
polarized                        0
profanity_obscenity              0
threat_incitement_to_violence    0
insults                          0
identity_attack                  0
sexually_explicit                0
key                              0
cleaned_text                     0
processed_text                   0
dtype: int64

In [73]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26375 entries, 0 to 27832
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   text_id                        26375 non-null  object
 1   annotators_id                  26375 non-null  object
 2   text                           26375 non-null  object
 3   topic                          26375 non-null  object
 4   is_noise_or_spam_text          26375 non-null  int64 
 5   toxicity                       26375 non-null  int64 
 6   polarized                      26375 non-null  object
 7   profanity_obscenity            26375 non-null  int64 
 8   threat_incitement_to_violence  26375 non-null  int64 
 9   insults                        26375 non-null  int64 
 10  identity_attack                26375 non-null  int64 
 11  sexually_explicit              26375 non-null  int64 
 12  key                            26375 non-null  object
 13  cleane

In [74]:
df_main.head(1)

,text_id,annotators_id,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,key,cleaned_text,processed_text
0,1-1,"['7', '15']",Wajah Jurnalis Dibalut Perban Saat Melaporkan Kondisi di Gaza 2023,UNKNOWN,0,0,"[0, 0]",0,0,0,0,0,"1-1_['7', '15']",Wajah jurnalis dibalut perban saat melaporkan kondisi di Gaza 2023,wajah jurnalis dibalut perban saat melaporkan kondisi di gaza


### PROCESS POLARIZED USING AGG (LIKE PAPER)

In [76]:
x = df_main["polarized"].iloc[0]
print(type(x), repr(x))

<class 'list'> ['0', '0']


In [77]:
import numpy as np

df_main["polarization_list"] = df_main["polarized"].apply(lambda xs: [int(v) for v in xs])
df_main["pol_avg"] = df_main["polarization_list"].apply(lambda xs: float(np.mean(xs)))

print(df_main[["polarized", "pol_avg"]].head())

   polarized   pol_avg
0     [0, 0]  0.000000
1     [1, 1]  1.000000
2  [0, 0, 0]  0.000000
3  [1, 0, 0]  0.333333
4     [1, 1]  1.000000


In [79]:
df_main.head(5)

,text_id,annotators_id,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,key,cleaned_text,processed_text,polarization_list,pol_avg
0,1-1,"['7', '15']",Wajah Jurnalis Dibalut Perban Saat Melaporkan Kondisi di Gaza 2023,UNKNOWN,0,0,"[0, 0]",0,0,0,0,0,"1-1_['7', '15']",Wajah jurnalis dibalut perban saat melaporkan kondisi di Gaza 2023,wajah jurnalis dibalut perban saat melaporkan kondisi di gaza,"[0, 0]",0.000000
1,1-10,"['7', '15']","Elektabilitas Paslon 02 sebentar lagi terjun bebas gara2 Gemoy, joget, susu gratis, makan gratis, politik dinasti, cacat hukum, Ijazah dan tidak hadir",Disabilitas,0,0,"[1, 1]",0,0,0,0,0,"1-10_['7', '15']","Elektabilitas pasangan calon 02 sebentar lagi terjun bebas gara-gara Gemoy, joget, susu gratis, makan gratis, politik dinasti, cacat hukum, ijazah, dan tidak hadir","elektabilitas pasangan calon sebentar lagi terjun bebas garagara gemoy, joget, susu gratis, makan gratis, politik dinasti, cacat hukum, ijazah, dan tidak hadir","[1, 1]",1.000000
2,1-100,"['7', '15', '32']","Gini aja deh @KPU_ID usul aja nih. Gimana kalo misalnya gak usah ada debat aja kalo ga boleh dikritik ? Diubah jadi lomba joget siapa yang paling GEndut SeMpOYongan 😂🤦👎 Menarik gak ? Atur lah, kalian kan penguasa.. 😁",Disabilitas,0,1,"[0, 0, 0]",0,0,1,0,0,"1-100_['7', '15', '32']","Gini aja deh @KPU_ID, usul aja nih. Gimana kalau misalnya tidak usah ada debat kalau nggak boleh dikritik? Diubah jadi lomba joget siapa yang paling gendut sempoyongan 😂🤦👎 Menarik nggak? Atur lah, kalian kan penguasa.. 😁","gini saja deh , usul saja nih. gimana kalau misalnya tidak usah ada debat kalau tidak boleh dikritik? diubah jadi lomba joget siapa yang paling gendut sempoyongan menarik nggak? atur lah, kalian kan penguasa..","[0, 0, 0]",0.000000
3,1-1000,"['7', '15', '32']",Relawan Tionghoa Kalbar yg di Jakarta mendeklarasikan dukungan kepada Ganjar Pranowo. Mario Dandy Bu D Megumi Putri Ariani Mamat Cilacap Mahkamah Konstitusi,Tionghoa,0,0,"[1, 0, 0]",0,0,0,0,0,"1-1000_['7', '15', '32']","Relawan Tionghoa Kalbar yang di Jakarta mendeklarasikan dukungan kepada Ganjar Pranowo. Nama terkait: Mario Dandy, Bu D, Megumi, Putri Ariani, Mamat. Cilacap. Mahkamah Konstitusi.","relawan tionghoa kalbar yang di jakarta mendeklarasikan dukungan kepada ganjar pranowo. nama terkait mario dandy, bu d, megumi, putri ariani, mamat. cilacap. mahkamah konstitusi.","[1, 0, 0]",0.333333
4,1-1001,"['10', '18']","Kristen, Hindu, Islam dapat perlakuan istimewa dari pak Anies Ncep ketar-ketir",Kristen,0,0,"[1, 1]",0,0,0,0,0,"1-1001_['10', '18']","Kristen, Hindu, Islam dapat perlakuan istimewa dari Pak Anies Ncep, ketar-ketir","kristen, hindu, islam dapat perlakuan istimewa dari pak anies ncep, ketarketir","[1, 1]",1.000000


## PREPROCESSING

In [80]:
def preprocess_tweet(text):
    if not isinstance(text, str):
        return ""

    text = emoji.demojize(text).lower()
     # 1. remove username (@user)
    text = re.sub(r'@\w+', '', text)

    # 2. remove URL
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 3. remove numbers
    text = re.sub(r'\d+', '', text)

    # 4. remove unwanted symbols (keep .,!?)
    text = re.sub(r'[^\w\s.,!?]', '', text)

    # 6. normalize repeated chars (contoh: anjiiing → anjiing)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 7. lowercase (optional)
    text = text.lower()

    # 8. remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [81]:
from tqdm import tqdm
tqdm.pandas()

df_main["processed_text"] = (
    df_main["cleaned_text"]
    .fillna("")
    .astype(str)
    .progress_apply(
        lambda x: preprocess_tweet(x)
    )
)

100%|██████████| 26375/26375 [00:06<00:00, 3817.45it/s]


In [82]:
df_main['toxicity'].value_counts()

toxicity
0    23976
1     2399
Name: count, dtype: int64

In [83]:
df_main.head(2)

,text_id,annotators_id,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,key,cleaned_text,processed_text,polarization_list,pol_avg
0,1-1,"['7', '15']",Wajah Jurnalis Dibalut Perban Saat Melaporkan Kondisi di Gaza 2023,UNKNOWN,0,0,"[0, 0]",0,0,0,0,0,"1-1_['7', '15']",Wajah jurnalis dibalut perban saat melaporkan kondisi di Gaza 2023,wajah jurnalis dibalut perban saat melaporkan kondisi di gaza,"[0, 0]",0.0
1,1-10,"['7', '15']","Elektabilitas Paslon 02 sebentar lagi terjun bebas gara2 Gemoy, joget, susu gratis, makan gratis, politik dinasti, cacat hukum, Ijazah dan tidak hadir",Disabilitas,0,0,"[1, 1]",0,0,0,0,0,"1-10_['7', '15']","Elektabilitas pasangan calon 02 sebentar lagi terjun bebas gara-gara Gemoy, joget, susu gratis, makan gratis, politik dinasti, cacat hukum, ijazah, dan tidak hadir","elektabilitas pasangan calon sebentar lagi terjun bebas garagara gemoy, joget, susu gratis, makan gratis, politik dinasti, cacat hukum, ijazah, dan tidak hadir","[1, 1]",1.0


# TRY BALANCING 1:3 RATIO (LIKE PAPER) -- UNDERSAMPLING

In [84]:
pos = df_main[df_main["toxicity"] == 1]
neg = df_main[df_main["toxicity"] == 0].sample(n=3 * len(pos), random_state=42)
df_final = pd.concat([pos, neg]).sample(frac=1, random_state=42).reset_index(drop=True)

In [85]:
df_final['toxicity'].value_counts()

toxicity
0    7197
1    2399
Name: count, dtype: int64

In [86]:
df_final.head(10)

,text_id,annotators_id,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,key,cleaned_text,processed_text,polarization_list,pol_avg
0,2-7654,['11'],"kok bisa ya ada orang yang dukung israel kalo dulu kita dijajah belanda ada gak ya yang dukung belanda malah orang israel sendiri aja ada yg dukung palestina, coba liat palestina mana ada yg dukung israel, dr sini tau kan siapa yg salah🙏",Jewish,0,1,[1],0,0,0,1,0,2-7654_['11'],"Kenapa ada orang yang dukung Israel kalau dulu kita dijajah Belanda? Ada tidak ya yang dukung Belanda? Malah orang Israel sendiri ada yang dukung Palestina. Coba lihat, Palestina mana ada yang dukung Israel. Dari sini tahu kan siapa yang salah🙏","kenapa ada orang yang dukung israel kalau dulu kita dijajah belanda? ada tidak ya yang dukung belanda? malah orang israel sendiri ada yang dukung palestina. coba lihat, palestina mana ada yang dukung israel. dari sini tahu kan siapa yang salahfolded_hands",[1],1.000000
1,1-430,"['7', '15']","Stigma anti anies komunis itu senjatanya mereka untuk menakut2i pihak yang selama ini kritis pada anies jadi takut kritik anies,,modusnya mereka bisa dilihat secara kasat mata mereka senantiasa membabi buta melabeli komunis pihak yang kritis pada anies.☠️",Disabilitas,0,0,"[0, 0]",0,0,0,0,0,"1-430_['7', '15']",Stigma anti-Anies 'komunis' itu senjatanya; mereka gunakan itu untuk menakut-nakuti pihak yang selama ini kritis pada Anies sehingga takut mengkritik Anies. Modusnya bisa dilihat secara kasat mata: mereka senantiasa membabi buta melabeli komunis pihak yang kritis pada Anies. ☠️,stigma antianies komunis itu senjatanya mereka gunakan itu untuk menakutnakuti pihak yang selama ini kritis pada anies sehingga takut mengkritik anies. modusnya bisa dilihat secara kasat mata mereka senantiasa membabi buta melabeli komunis pihak yang kritis pada anies. skull_and_crossbones,"[0, 0]",0.000000
2,2-6367,"['7', '18']",Sentimen The Fed dan Data Ekspor-Impor China Tekan IHSG,Tionghoa,0,0,"[0, 0]",0,0,0,0,0,"2-6367_['7', '18']",Sentimen The Fed dan data ekspor-impor China tekan IHSG,sentimen the fed dan data eksporimpor china tekan ihsg,"[0, 0]",0.000000
3,2-5889,"['1', '18']","MEMBACA FREUD TENTANG ALAM BAWAH SADAR Refleksi pembiasaan bentuk pola sikap ... (DOKTRIN KONSERVATIF) Setiap manusia terbentuk oleh langkah jejak Alam keseharian turut mengikat Orang orang dekat membekali watak Darah nutrisi amat sangat mencipta harkat (DOKTRIN LIBERALISTIK) Logika empiris mengiringi detak nafas diri Bentuk pengalaman dan bangun pondasi persepsi Persepsi sikap dini respon pemula yang bersifat spontan dan mandiri Pengalaman masa pertumbuhan bentuk dasar peranggai dan pola sikap Tapak jejak perkembangan pengaruhi pola pikir dan peranggai diri Pengalaman yang membiasa panjang dan hiasi langkahnya, membentuk alam bawah sadar yang menjiwa kuat Itulah kerangka konsep diri sebagai dasar cara diri memposisikan harkat dan martabat Alam bawah sadar Situasi daya potensial yang menjiwa Selalu jadi pola pola prilaku hadapi setiap dinamika diri '𝗣𝘀𝘆𝗰𝗵𝗼𝗽𝗮𝘁𝗼𝗹𝗼𝗴𝘆 𝗼𝗳 𝗘𝘃𝗲𝗿𝘆𝗱𝗮𝘆 𝗟𝗶𝗳𝗲' 𝗞𝗮𝗿𝘆𝗮 𝗦𝗶𝗴𝗺𝘂𝗻𝗱 𝗙𝗿𝗲𝘂𝗱! Karya paling populer, Sigmund Freud menuangkan gagasan-gagasannya mengenai pengalaman bentuk alam bawah sadar Kealaman masa lalu, akan menjadi kesalahan atau kebenaran yang terjadi dalam kehidupan Jiwa yang terbiasa diisi ""a"" Akan membentuk prilaku serupa Buah akan jatuh di sekitar pohon Ikut menabur angin akan menubai badai Isilah alam bawah sadar dengan ilmu iman dan amal (DOKTRIN KRITIS ILMIYAH) Pembentuk alam bawah sadar menjadi pondasi karakter diri Akibat sentuhan kehidupan pribadi, kolega, keluarga, lingkup sosial dan pasien karyawannya Lewat pendekatan psikoanalisis, Freud menemukan betapa besar pengalaman bentuk alam bawah sadar Jejak pengalaman akan beri garis pemisah antara perilaku normal dan perilaku neurotik diri Neurotik ekspresi prilaku tidak normal, kehilangan jesadaran diri, dan ketidak mampuan diri jalani 

# BASELINE
epoch = 5
batch = 8
learning rate = 1e-5
weight_decay = 0.01

In [ ]:
# def add_polarization_text(row):
#     return f"[POL={row['polarized']}] {row['processed_text']}"

# df_main["text_with_pol"] = df_main.apply(add_polarization_text, axis=1)

In [87]:
x = df_final[["processed_text", "pol_avg"]]
y = df_final['toxicity']

In [88]:
x_train_full, x_test, y_train_full, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.125, stratify=y_train_full, random_state=42
)

print("Stage 1 split sizes:", {"train": len(x_train), "val": len(x_val), "test": len(x_test)})

Stage 1 split sizes: {'train': 6716, 'val': 960, 'test': 1920}


In [89]:
class_weights_s1 = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights_s1 = torch.tensor(class_weights_s1, dtype=torch.float)
print("Stage 1 class weights (class 0, class 1):", class_weights_s1)

Stage 1 class weights (class 0, class 1): tensor([0.6667, 2.0000])


In [94]:
from datasets import Dataset, DatasetDict

def make_df(x, y):
    df = x.copy()
    df["label"] = y.values
    return df.reset_index(drop=True)

train_df = make_df(x_train, y_train)
val_df = make_df(x_val, y_val)
test_df = make_df(x_test, y_test)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['processed_text', 'pol_avg', 'label'],
        num_rows: 6716
    })
    validation: Dataset({
        features: ['processed_text', 'pol_avg', 'label'],
        num_rows: 960
    })
    test: Dataset({
        features: ['processed_text', 'pol_avg', 'label'],
        num_rows: 1920
    })
})


In [95]:
from transformers import (
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
TrainerCallback
)
tokenizer = AutoTokenizer.from_pretrained("indolem/indobertweet-base-uncased")

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [105]:
# def tokenize(batch):
#     return tokenizer(batch["processed_text"], truncation=True, max_length=128, padding='max_length')

def tokenize_with_feature(batch):
    prefix = [f"Nilai rata-rata polarisasi: {v:.2f}" for v in batch["pol_avg"]]
    return tokenizer(prefix, batch["processed_text"], truncation="only_second", max_length=128, padding='max_length')

tokenized = dataset.map(tokenize_with_feature, batched=True)
tokenized = tokenized.remove_columns(["processed_text", "pol_avg"])

print(tokenizer.decode(tokenized["train"][0]["input_ids"]))

Map:   0%|          | 0/6716 [00:00<?, ? examples/s]

Map:   0%|          | 0/960 [00:00<?, ? examples/s]

Map:   0%|          | 0/1920 [00:00<?, ? examples/s]

[CLS] nilai rata - rata polarisasi : 0. 00 [SEP] klasemen akhir asian games hangzhou indonesia finis posisi, china rekor medali emas. woman _ lifting _ weights cabang olahraga peraih medali untuk indonesia st _ place _ medal atlet peraih emas nd _ place _ medal atlet peraih perak rd _ place _ medal atlet peraih perunggu selengkapnya [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [106]:
tokenized_datasets_1 = dataset.map(tokenize_with_feature, batched=True)

tokenized_datasets_1 = tokenized_datasets_1.remove_columns(["processed_text",'pol_avg'])
tokenized_datasets_1 = tokenized_datasets_1.rename_column("label", "labels")
tokenized_datasets_1.set_format("torch")

train_dataset_1 = tokenized_datasets_1["train"]
val_dataset_1 = tokenized_datasets_1["validation"]
test_dataset_1 = tokenized_datasets_1["test"]

Map:   0%|          | 0/6716 [00:00<?, ? examples/s]

Map:   0%|          | 0/960 [00:00<?, ? examples/s]

Map:   0%|          | 0/1920 [00:00<?, ? examples/s]

In [107]:
    import numpy as np
    from scipy.special import softmax
    from sklearn.metrics import (
        accuracy_score, f1_score, precision_score, recall_score,
        roc_auc_score, average_precision_score,
    )
    
    def compute_metrics_s1(eval_pred):
        logits, labels = eval_pred
        probs = softmax(logits, axis=1)[:, 1]
        preds = logits.argmax(axis=1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "precision": precision_score(labels, preds, zero_division=0),
            "recall": recall_score(labels, preds, zero_division=0),
            "f1": f1_score(labels, preds, zero_division=0),
            "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
            "roc_auc": roc_auc_score(labels, probs),
            "pr_auc": average_precision_score(labels, probs),
        }


In [108]:
from transformers import AutoModelForSequenceClassification

model_s1 = AutoModelForSequenceClassification.from_pretrained(
    "indolem/indobertweet-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [109]:
# Trainer that applies class weights to CrossEntropyLoss, used to handle label
# imbalance in Stage 1.
class WeightedTrainerSingleLabel(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = nn.CrossEntropyLoss(weight=weight)
        # logits.size(-1) instead of model.config.num_labels: stays correct even if
        # Trainer wraps the model in nn.DataParallel (which doesn't expose .config).
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [110]:
from transformers import TrainingArguments, Trainer

args_s1 = TrainingArguments(
    output_dir="./model_toxicity",
    num_train_epochs=5,
    logging_steps=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate = 1e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)


trainer_s1 = WeightedTrainerSingleLabel(
    model=model_s1,
    args=args_s1,
    train_dataset=train_dataset_1,
    eval_dataset=val_dataset_1,
    compute_metrics=compute_metrics_s1,
    class_weights=class_weights_s1,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer_s1.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Macro,Roc Auc,Pr Auc
1,0.245345,0.450371,0.758333,0.509479,0.895833,0.649547,0.732564,0.893038,0.737373
2,0.179790,0.426712,0.807292,0.578348,0.845833,0.686971,0.773884,0.900191,0.758718
3,0.230963,0.441871,0.815625,0.591304,0.850000,0.697436,0.782426,0.901742,0.766353
4,0.179582,0.505473,0.850000,0.670213,0.787500,0.724138,0.810567,0.903478,0.774345


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1680, training_loss=0.3668155218663049, metrics={'train_runtime': 518.9153, 'train_samples_per_second': 64.712, 'train_steps_per_second': 4.047, 'total_flos': 1767053847797760.0, 'train_loss': 0.3668155218663049, 'epoch': 4.0})

In [111]:
from sklearn.metrics import classification_report
pred1 = trainer_s1.predict(test_dataset_1)
pred_label1 = pred1.predictions.argmax(axis=1)

probs1_test = torch.softmax(torch.tensor(pred1.predictions), dim=-1)[:, 1].numpy()
# pred_label1_tuned = (probs1_test >= best_threshold_s1).astype(int)

print("Stage 1 classification report @ default threshold (0.5) -- test set:")
print(classification_report(y_test, pred_label1))

# print(f"\nStage 1 classification report @ tuned threshold ({best_threshold_s1:.4f}) -- test set:")
# print(classification_report(y_test, pred_label1_tuned))

Stage 1 classification report @ default threshold (0.5) -- test set:
              precision    recall  f1-score   support

           0       0.94      0.80      0.87      1440
           1       0.59      0.86      0.70       480

    accuracy                           0.82      1920
   macro avg       0.77      0.83      0.78      1920
weighted avg       0.86      0.82      0.83      1920

